In [17]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import ipywidgets as widgets
from IPython.display import display

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass


In [18]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
np.random.seed(0)
torch.manual_seed(0)


In [19]:
def length(p):
    return np.linalg.norm(p, axis=-1)

def sd_circle(p, r):
    return length(p) - r

def sd_box(p, b):
    q = np.abs(p) - np.asarray(b)
    return length(np.maximum(q, 0.0)) + np.minimum(np.maximum(q[..., 0], q[..., 1]), 0.0)

def sd_rounded_box(p, b, r):
    q = np.abs(p) - np.asarray(b) + r
    return length(np.maximum(q, 0.0)) + np.minimum(np.maximum(q[..., 0], q[..., 1]), 0.0) - r

def sd_segment(p, a, b):
    a = np.asarray(a)
    b = np.asarray(b)
    pa = p - a
    ba = b - a
    h = np.clip(np.sum(pa * ba, axis=-1) / np.dot(ba, ba), 0.0, 1.0)
    return length(pa - h[..., None] * ba)

def sd_capsule(p, a, b, r):
    return sd_segment(p, a, b) - r

def ndot(a, b):
    return a[..., 0] * b[..., 0] - a[..., 1] * b[..., 1]

def sd_rhombus(p, b):
    b = np.asarray(b)
    p = np.abs(p)
    h = np.clip((-2.0 * ndot(p, b) + ndot(b, b)) / np.dot(b, b), -1.0, 1.0)
    q = p - 0.5 * b * np.stack((1.0 - h, 1.0 + h), axis=-1)
    s = np.sign(p[..., 0] * b[1] + p[..., 1] * b[0] - b[0] * b[1])
    return length(q) * s

def sd_equilateral_triangle(p, r):
    k = np.sqrt(3.0)
    x = np.abs(p[..., 0]) - r
    y = p[..., 1] + r / k
    fold = x + k * y > 0.0
    tx = 0.5 * (x - k * y)
    ty = 0.5 * (-k * x - y)
    x = np.where(fold, tx, x)
    y = np.where(fold, ty, y)
    x -= np.clip(x, -2.0 * r, 0.0)
    return -np.hypot(x, y) * np.sign(y)

def sd_hexagon(p, r):
    k = np.array([-0.8660254038, 0.5, 0.5773502692])
    q = np.abs(p)
    h = np.minimum(q[..., 0] * k[0] + q[..., 1] * k[1], 0.0)
    q = q - 2.0 * h[..., None] * k[:2]
    q = q - np.stack((np.clip(q[..., 0], -k[2] * r, k[2] * r), np.full(q.shape[:-1], r)), axis=-1)
    return length(q) * np.sign(q[..., 1])

def sd_octagon(p, r):
    k = np.array([-0.9238795325, 0.3826834323, 0.4142135624])
    q = np.abs(p)
    n0 = k[:2]
    h0 = np.minimum(np.sum(q * n0, axis=-1), 0.0)
    q = q - 2.0 * h0[..., None] * n0
    n1 = np.array([-k[0], k[1]])
    h1 = np.minimum(np.sum(q * n1, axis=-1), 0.0)
    q = q - 2.0 * h1[..., None] * n1
    q = q - np.stack((np.clip(q[..., 0], -k[2] * r, k[2] * r), np.full(q.shape[:-1], r)), axis=-1)
    return length(q) * np.sign(q[..., 1])


In [20]:
SDFS = {
    "Circle": lambda p: sd_circle(p, 0.65),
    "Box": lambda p: sd_box(p, (0.72, 0.48)),
    "Rounded box": lambda p: sd_rounded_box(p, (0.76, 0.52), 0.16),
    "Capsule": lambda p: sd_capsule(p, (-0.65, -0.35), (0.55, 0.45), 0.16),
    "Rhombus": lambda p: sd_rhombus(p, (0.75, 0.55)),
    "Equilateral triangle": lambda p: sd_equilateral_triangle(p, 0.72),
    "Hexagon": lambda p: sd_hexagon(p, 0.68),
    "Octagon": lambda p: sd_octagon(p, 0.68),
}


In [21]:
class Transform:
    def __init__(self, position=(0.0, 0.0), rotation=0.0, scale=1.0):
        self.position = np.asarray(position, dtype=np.float32)
        self.rotation = float(rotation)
        self.scale = float(scale)
        if self.position.shape != (2,) or not np.isfinite(self.position).all():
            raise ValueError("Position must contain two finite values")
        if not np.isfinite(self.rotation) or not np.isfinite(self.scale) or self.scale <= 0:
            raise ValueError("Rotation must be finite and uniform scale must be positive")

    def matrix(self):
        cosine = np.cos(self.rotation)
        sine = np.sin(self.rotation)
        return np.array([[cosine, -sine], [sine, cosine]], dtype=np.float32)

    def to_world(self, points):
        return np.asarray(points) @ self.matrix().T * self.scale + self.position

    def to_local(self, points):
        return ((np.asarray(points) - self.position) / self.scale) @ self.matrix()


In [22]:
class ParticleBuffer:
    def __init__(self):
        self.positions = np.empty((0, 2), dtype=np.float32)
        self.velocities = np.empty((0, 2), dtype=np.float32)

    def push(self, positions, velocities=None):
        positions = np.asarray(positions, dtype=np.float32)
        if positions.ndim != 2 or positions.shape[1] != 2 or not np.isfinite(positions).all():
            raise ValueError("Particle positions must have shape (N, 2) and be finite")

        if velocities is None:
            velocities = np.zeros_like(positions)
        else:
            velocities = np.asarray(velocities, dtype=np.float32)

        if velocities.shape != positions.shape or not np.isfinite(velocities).all():
            raise ValueError("Velocities must match particle positions")

        self.positions = np.concatenate(
            (self.positions, positions)
        )

        self.velocities = np.concatenate(
            (self.velocities, velocities)
        )

    def step(self, dt):
        self.positions += self.velocities * dt


In [23]:
class ParticleModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(2, 64),
            nn.Softplus(beta=10),
            nn.Linear(64, 128),
            nn.Softplus(beta=10)
        )

        self.decoder = nn.Sequential(
            nn.Linear(130, 128),
            nn.Softplus(beta=10),
            nn.Linear(128, 64),
            nn.Softplus(beta=10),
            nn.Linear(64, 1)
        )

    def forward(self, particle_points, query_points):
        if particle_points.shape[1] == 0:
            raise ValueError("No particles: increase the particle resolution")
        point_features = self.encoder(particle_points)
        shape_code = point_features.max(dim=1).values

        shape_code = shape_code[:, None, :].expand(
            -1,
            query_points.shape[1],
            -1
        )

        decoder_input = torch.cat(
            (query_points, shape_code),
            dim=-1
        )

        return self.decoder(decoder_input)


In [24]:
import random
torch.manual_seed(0)
class SDFObject:
    def __init__(self, sdf, position=(0.0, 0.0), NN=False, rotation=0.0, scale=1.0, name=None, extent=1.5):
        self.sdf = sdf
        self.transform = Transform(position, rotation, scale)
        self.name = name or "Shape"
        self.extent = float(extent)
        if not np.isfinite(self.extent) or self.extent <= 0:
            raise ValueError("Extent must be positive")
        self.particles = ParticleBuffer()
        self.NN = NN
        self.model = nn.Sequential(
            nn.Linear(2, 64),
            nn.Softplus(beta=10),
            nn.Linear(64, 64),
            nn.Softplus(beta=10),
            nn.Linear(64, 1)
        ).to(device)
        self.particle_model = None

    def sample(self, points):
        local_points = self.transform.to_local(points)

        if self.NN:
            return self.sampleNN(local_points) * self.transform.scale

        return self.sdf(local_points) * self.transform.scale

    def sampleNN(self, points):
        device = next(self.model.parameters()).device

        point_tensor = torch.as_tensor(
            points,
            dtype=torch.float32,
            device=device
        )

        self.model.eval()

        with torch.no_grad():
            distance = self.model(point_tensor)

        return distance.squeeze(-1).cpu().numpy()

    def sampleParticleNN(self, particles, points):
        if len(particles.positions) == 0:
            raise ValueError("No particles: increase the particle resolution")
        query_shape = points.shape[:-1]

        local_query_points = (
            self.transform.to_local(points)
        ).reshape(-1, 2)

        local_particle_points = (
            particles.positions
        )

        if self.particle_model is None:
            raise RuntimeError("Assign or train a particle model before sampling")
        device = next(
            self.particle_model.parameters()
        ).device

        query_tensor = torch.as_tensor(
            local_query_points,
            dtype=torch.float32,
            device=device
        ).unsqueeze(0)

        particle_tensor = torch.as_tensor(
            local_particle_points,
            dtype=torch.float32,
            device=device
        ).unsqueeze(0)

        if self.particle_model is None:
            raise RuntimeError("Assign or train a particle model before sampling")
        self.particle_model.eval()

        with torch.no_grad():
            distance = self.particle_model(
                particle_tensor,
                query_tensor
            )

        return self.transform.scale * (
            distance[0, :, 0]
            .reshape(query_shape)
            .cpu()
            .numpy()
        )

    def train(self, particles=None):
        if particles is None:
            return self._train_source_model()

        return self._train_particle_model(particles)


    def _train_source_model(self):
        self.model.train()

        device = next(
            self.model.parameters()
        ).device

        optimizer = torch.optim.Adam(
            self.model.parameters(),
            lr=1e-3
        )

        loss_function = nn.MSELoss()

        for step in range(2000):
            local_points_np = np.random.uniform(
                -self.extent,
                self.extent,
                size=(4096, 2)
            )

            target_distances_np = self.sdf(
                local_points_np
            )

            local_points = torch.as_tensor(
                local_points_np,
                dtype=torch.float32,
                device=device
            )

            target_distances = torch.as_tensor(
                target_distances_np,
                dtype=torch.float32,
                device=device
            ).unsqueeze(-1)

            optimizer.zero_grad(set_to_none=True)

            predicted_distances = self.model(
                local_points
            )

            loss = loss_function(
                predicted_distances,
                target_distances
            )

            loss.backward()
            optimizer.step()

            if step % 100 == 0:
                print("source", step, loss.item())


    def _train_particle_model(self, particles):
        if self.particle_model is None:
            self.particle_model = ParticleModel().to(next(self.model.parameters()).device)
        self.particle_model.train()

        if self.particle_model is None:
            raise RuntimeError("Assign or train a particle model before sampling")
        device = next(
            self.particle_model.parameters()
        ).device

        particle_points_np = (
            particles.positions
        )

        particle_points = torch.as_tensor(
            particle_points_np,
            dtype=torch.float32,
            device=device
        ).unsqueeze(0)

        optimizer = torch.optim.Adam(
            self.particle_model.parameters(),
            lr=1e-3
        )

        loss_function = nn.MSELoss()

        for step in range(2000):
            query_points_np = np.random.uniform(
                -self.extent,
                self.extent,
                size=(4096, 2)
            )

            target_distances_np = self.sdf(
                query_points_np
            )

            query_points = torch.as_tensor(
                query_points_np,
                dtype=torch.float32,
                device=device
            ).unsqueeze(0)

            target_distances = torch.as_tensor(
                target_distances_np,
                dtype=torch.float32,
                device=device
            ).unsqueeze(0).unsqueeze(-1)

            optimizer.zero_grad(set_to_none=True)

            predicted_distances = self.particle_model(
                particle_points,
                query_points
            )

            loss = loss_function(
                predicted_distances,
                target_distances
            )

            loss.backward()
            optimizer.step()

            if step % 100 == 0:
                print("particles", step, loss.item())

def push_sdf(obj):
    sdf_buffer.append(obj)
    return obj


In [25]:
class ParticleSystem:
    def __init__(self):
        self.objects = []

    def sample(self, obj, resolution=31, spacing=None, extent=None):
        extent = obj.extent if extent is None else float(extent)
        if not np.isfinite(extent) or extent <= 0:
            raise ValueError("Extent must be positive")

        if spacing is None:
            counts = np.asarray(resolution)
            if counts.ndim == 0:
                counts = np.repeat(counts, 2)
            if counts.shape != (2,) or not np.isfinite(counts).all():
                raise ValueError("Resolution must be an integer or a pair of integers")
            if np.any(counts < 2) or np.any(counts != np.floor(counts)):
                raise ValueError("Resolution must be at least two nodes per axis")
            x = np.linspace(-extent, extent, int(counts[0]))
            y = np.linspace(-extent, extent, int(counts[1]))
        else:
            spacing = np.asarray(spacing, dtype=float)
            if spacing.ndim == 0:
                spacing = np.repeat(spacing, 2)
            if spacing.shape != (2,) or not np.isfinite(spacing).all() or np.any(spacing <= 0):
                raise ValueError("Spacing must be a positive value or pair")
            x = np.arange(-extent, extent + spacing[0] * 0.5, spacing[0])
            y = np.arange(-extent, extent + spacing[1] * 0.5, spacing[1])

        grid_x, grid_y = np.meshgrid(x, y)
        points = np.column_stack((grid_x.ravel(), grid_y.ravel())).astype(np.float32)
        distances = np.asarray(obj.sdf(points))
        if distances.shape != (len(points),) or not np.isfinite(distances).all():
            raise ValueError("The shape must return one finite distance per point")

        particles = ParticleBuffer()
        particles.push(points[distances <= 0.0])
        return particles

    def create_object(self, sdf, name=None, position=(0.0, 0.0), rotation=0.0,
                      scale=1.0, resolution=31, spacing=None, extent=1.5, NN=False):
        obj = SDFObject(sdf, position, NN, rotation, scale, name, extent)
        obj.particles = self.sample(obj, resolution, spacing, extent)
        self.objects.append(obj)
        return obj

    def resample(self, obj, resolution=31, spacing=None, extent=None):
        obj.particles = self.sample(obj, resolution, spacing, extent)
        return obj.particles

    def step(self, dt):
        for obj in self.objects:
            obj.particles.step(dt)


def create_particle_buffer(obj, extent=None, lattice_resolution=31, spacing=None):
    return particle_system.sample(obj, lattice_resolution, spacing, extent)


In [26]:
from matplotlib.path import Path

def make_polygon_sdf(vertices):
    vertices = np.asarray(vertices, dtype=float)

    segment_start = vertices
    segment_end = np.roll(vertices, -1, axis=0)
    segment_vector = segment_end - segment_start
    segment_length_squared = np.sum(
        segment_vector * segment_vector,
        axis=1
    )

    polygon = Path(np.vstack((vertices, vertices[0])), closed=True)

    def sdf(points):
        original_shape = points.shape[:-1]
        points = np.asarray(points, dtype=float).reshape(-1, 2)

        relative = (
            points[:, None, :]
            - segment_start[None, :, :]
        )

        t = np.sum(
            relative * segment_vector[None, :, :],
            axis=-1
        ) / segment_length_squared[None, :]

        t = np.clip(t, 0.0, 1.0)

        closest_points = (
            segment_start[None, :, :]
            + t[:, :, None] * segment_vector[None, :, :]
        )

        displacement = (
            points[:, None, :]
            - closest_points
        )

        distance = np.sqrt(
            np.min(
                np.sum(displacement * displacement, axis=-1),
                axis=1
            )
        )

        inside = polygon.contains_points(points)
        distance[inside] *= -1.0

        return distance.reshape(original_shape)

    return sdf


angles = np.linspace(0.0, 2.0 * np.pi, 12, endpoint=False)

radii = np.array([
    0.82, 0.70, 0.78, 0.68,
    0.85, 0.73, 0.80, 0.69,
    0.76, 0.86, 0.71, 0.79
])

blob_vertices = np.stack(
    (
        radii * np.cos(angles),
        radii * np.sin(angles)
    ),
    axis=-1
)


In [27]:
particle_resolution = 31
particle_spacing = None
training_resolutions = [12, 20, 31, 48]
training_steps = 35000
query_count = 4096


In [28]:
particle_system = ParticleSystem()

objects = [
    particle_system.create_object(
        sdf,
        name=name,
        resolution=particle_resolution,
        spacing=particle_spacing
    )
    for name, sdf in SDFS.items()
]

circle = objects[0]
box = objects[1]
circle.NN = True
box.NN = True
box.transform.position = np.array((0.6, 0.0), dtype=np.float32)

blob = particle_system.create_object(
    make_polygon_sdf(blob_vertices),
    name="Blob",
    resolution=particle_resolution,
    spacing=particle_spacing
)
objects.append(blob)

sdf_buffer = [circle, box]

for obj in objects:
    print(obj.name, len(obj.particles.positions))


Circle 137
Box 135
Rounded box 161
Capsule 60
Rhombus 85
Equilateral triangle 95
Hexagon 155
Octagon 145
Blob 175


In [29]:
for obj in (circle, box):
    obj.train()


source 0 0.4954436123371124
source 100 0.006857908330857754
source 200 0.0015702806413173676
source 300 0.0007719302666373551
source 400 0.0004186934093013406
source 500 0.0002725252415984869
source 600 0.0001765600754879415
source 700 0.0001634584623388946
source 800 0.00011091776832472533
source 900 8.70483709149994e-05
source 1000 8.545467426301911e-05
source 1100 7.269725756486878e-05
source 1200 5.208741640672088e-05
source 1300 5.023120320402086e-05
source 1400 4.1701427107909694e-05
source 1500 4.216047454974614e-05
source 1600 3.0443254217971116e-05
source 1700 3.360255868756212e-05
source 1800 3.917136928066611e-05
source 1900 2.4021755962166935e-05
source 0 0.314420610666275
source 100 0.007097742520272732
source 200 0.0018915531691163778
source 300 0.0009619856718927622
source 400 0.0006309282034635544
source 500 0.00047208258183673024
source 600 0.0003815074451267719
source 700 0.00034492890699766576
source 800 0.0002909593458753079
source 900 0.00025407664361409843
source 

KeyboardInterrupt: 

In [ ]:
def plot_scene(extent=2.0, resolution=500):
    axis = np.linspace(-extent, extent, resolution)
    x, y = np.meshgrid(axis, axis)
    points = np.stack((x, y), axis=-1)

    distance = np.full(x.shape, np.inf)

    for obj in sdf_buffer:
        distance = np.minimum(distance, obj.sample(points))

    fig, ax = plt.subplots(figsize=(7, 7))

    ax.imshow(
        distance,
        extent=(-extent, extent, -extent, extent),
        origin="lower",
        cmap="RdBu_r",
    )

    ax.contour(x, y, distance, levels=[0.0], colors="black", linewidths=2.5)
    ax.set_aspect("equal")

    plt.show()

plot_scene()


In [ ]:
def sample_neural_sdf_lattice(obj, extent, spacing):
    axis = np.arange(
        -extent,
        extent + spacing * 0.5,
        spacing
    )

    x, y = np.meshgrid(axis, axis)

    local_points = np.stack(
        (x.ravel(), y.ravel()),
        axis=-1
    ).astype(np.float32)

    device = next(obj.model.parameters()).device

    point_tensor = torch.as_tensor(
        local_points,
        dtype=torch.float32,
        device=device
    )

    obj.model.eval()

    with torch.no_grad():
        distances = obj.model(point_tensor).squeeze(-1)

    inside = distances < 0.0

    inside_local_points = local_points[
        inside.cpu().numpy()
    ]

    return inside_local_points


def plot_particles(particles, transform=None):
    positions = particles.positions if transform is None else transform.to_world(particles.positions)

    if len(positions) == 0:
        raise RuntimeError("Particle buffer is empty")

    fig, ax = plt.subplots(figsize=(7, 7))

    ax.scatter(
        positions[:, 0],
        positions[:, 1],
        s=10,
        color="black"
    )

    padding = 0.25

    ax.set_xlim(
        positions[:, 0].min() - padding,
        positions[:, 0].max() + padding
    )

    ax.set_ylim(
        positions[:, 1].min() - padding,
        positions[:, 1].max() + padding
    )

    ax.set_aspect("equal")
    plt.show()


particles = ParticleBuffer()

sampled_positions = sample_neural_sdf_lattice(
    circle,
    extent=1.5,
    spacing=0.1
)

particles.push(sampled_positions)

plot_particles(particles, circle.transform)


In [ ]:
def sample_metaball_field(
    query_points,
    particles,
    sigma,
    threshold,
    batch_size=8192,
    transform=None
):
    particle_positions = torch.as_tensor(
        particles.positions if transform is None else transform.to_world(particles.positions),
        dtype=torch.float32,
        device=device
    )

    query_points = torch.as_tensor(
        query_points,
        dtype=torch.float32,
        device=device
    )

    field = torch.empty(
        len(query_points),
        dtype=torch.float32,
        device=device
    )

    with torch.no_grad():
        for start in range(0, len(query_points), batch_size):
            end = start + batch_size
            query_batch = query_points[start:end]

            displacement = (
                query_batch[:, None, :]
                - particle_positions[None, :, :]
            )

            squared_distance = displacement.square().sum(dim=-1)

            density = torch.exp(
                -squared_distance / (2.0 * sigma * sigma)
            ).sum(dim=1)

            field[start:end] = density

    return field.cpu().numpy()


In [ ]:
def plot_metaballs(
    particles,
    bounds,
    resolution,
    sigma,
    threshold,
    transform=None
):
    xmin, xmax, ymin, ymax = bounds

    x = np.linspace(xmin, xmax, resolution)
    y = np.linspace(ymin, ymax, resolution)

    grid_x, grid_y = np.meshgrid(x, y)

    query_points = np.stack(
        (grid_x.ravel(), grid_y.ravel()),
        axis=-1
    )

    field = sample_metaball_field(
        query_points,
        particles,
        sigma,
        threshold,
        transform=transform
    ).reshape(resolution, resolution)

    fig, ax = plt.subplots(figsize=(7, 7))

    ax.imshow(
        field,
        extent=bounds,
        origin="lower",
        cmap="RdBu_r"
    )

    positions = particles.positions if transform is None else transform.to_world(particles.positions)
    ax.scatter(
        positions[:, 0],
        positions[:, 1],
        s=5,
        color="black"
    )

    ax.set_aspect("equal")
    plt.show()


In [ ]:
spacing = 0.1
sigma = 0.12
threshold = np.pi * (sigma / spacing) ** 2

plot_metaballs(
    particles,
    bounds=(-2.0, 2.0, -2.0, 2.0),
    resolution=300,
    sigma=sigma,
    threshold=threshold,
    transform=circle.transform
)


In [ ]:
def create_training_example(obj, particles=None, extent=None, query_count=4096):
    particles = obj.particles if particles is None else particles
    extent = obj.extent if extent is None else float(extent)
    if len(particles.positions) == 0:
        raise ValueError(f"{obj.name} has no particles at this resolution")
    if not np.isfinite(extent) or extent <= 0:
        raise ValueError("Extent must be positive")
    if int(query_count) != query_count or query_count < 1:
        raise ValueError("Query count must be a positive integer")

    query_points = np.random.uniform(-extent, extent, (query_count, 2)).astype(np.float32)
    target_distances = np.asarray(obj.sdf(query_points), dtype=np.float32)
    if target_distances.shape != (query_count,) or not np.isfinite(target_distances).all():
        raise ValueError("The shape must return one finite distance per query")

    return particles.positions.copy(), query_points, target_distances


In [ ]:
def train_general_particle_model(
    model,
    objects,
    steps=35000,
    resolutions=None,
    query_count=4096
):
    if not objects:
        raise ValueError("Instantiate at least one training object")
    if resolutions is not None and len(resolutions) == 0:
        raise ValueError("Resolution choices cannot be empty")
    for obj in objects:
        obj.particle_model = model
    model.train()

    device = next(model.parameters()).device

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=1e-3
    )

    loss_function = nn.MSELoss()

    losses = []

    for step in range(steps):
        obj = objects[np.random.randint(len(objects))]
        particles = obj.particles
        if resolutions is not None:
            resolution = resolutions[np.random.randint(len(resolutions))]
            particles = particle_system.sample(obj, resolution=resolution)
        (
            particle_points_np,
            query_points_np,
            target_distances_np
        ) = create_training_example(obj, particles, query_count=query_count)

        particle_points = torch.as_tensor(
            particle_points_np,
            dtype=torch.float32,
            device=device
        ).unsqueeze(0)

        query_points = torch.as_tensor(
            query_points_np,
            dtype=torch.float32,
            device=device
        ).unsqueeze(0)

        target_distances = torch.as_tensor(
            target_distances_np,
            dtype=torch.float32,
            device=device
        ).unsqueeze(0).unsqueeze(-1)

        optimizer.zero_grad(set_to_none=True)

        predicted_distances = model(
            particle_points,
            query_points
        )

        loss = loss_function(
            predicted_distances,
            target_distances
        )

        loss.backward()
        optimizer.step()
        losses.append(loss.item())

        if step % 100 == 0:
            print(step, loss.item())
    return losses


In [ ]:
shared_particle_model = ParticleModel().to(device)


In [ ]:
losses = train_general_particle_model(
    shared_particle_model,
    objects,
    steps=training_steps,
    resolutions=training_resolutions,
    query_count=query_count
)


In [ ]:
def plot_particle_model(
    obj,
    particles,
    extent=2.0,
    resolution=500
):
    axis = np.linspace(-extent, extent, resolution)
    x, y = np.meshgrid(axis, axis)
    points = np.stack((x, y), axis=-1)

    distance = obj.sampleParticleNN(
        particles,
        points
    )

    fig, ax = plt.subplots(figsize=(7, 7))

    ax.imshow(
        distance,
        extent=(-extent, extent, -extent, extent),
        origin="lower",
        cmap="RdBu_r"
    )

    ax.set_aspect("equal")

    plt.show()


circle.particle_model = shared_particle_model
box.particle_model = shared_particle_model
blob.particle_model = shared_particle_model

circle_particles = particle_system.resample(circle, resolution=particle_resolution, spacing=particle_spacing)
box_particles = particle_system.resample(box, resolution=particle_resolution, spacing=particle_spacing)

plot_particle_model(
    circle,
    circle_particles
)

plot_particle_model(
    box,
    box_particles
)


In [ ]:
plot_metaballs(
    circle_particles,
    bounds=(-2.0, 2.0, -2.0, 2.0),
    resolution=500,
    sigma=0.12,
    threshold=0.0,
    transform=circle.transform
)

plot_particle_model(
    circle,
    circle_particles,
    extent=2.0,
    resolution=500
)

plot_metaballs(
    box_particles,
    bounds=(-2.0, 2.0, -2.0, 2.0),
    resolution=500,
    sigma=0.12,
    threshold=0.0,
    transform=box.transform
)

plot_particle_model(
    box,
    box_particles,
    extent=2.0,
    resolution=500
)


In [ ]:
def test_particle_dependence(
    obj,
    particles,
    extent=2.0,
    resolution=300
):
    axis = np.linspace(-extent, extent, resolution)
    x, y = np.meshgrid(axis, axis)
    query_points = np.stack((x, y), axis=-1)

    original_distance = obj.sampleParticleNN(
        particles,
        query_points
    )

    collapsed_particles = ParticleBuffer()
    collapsed_particles.push(
        particles.positions.copy()
    )

    collapsed_particles.positions[:] = (
        collapsed_particles.positions.mean(axis=0)
    )

    collapsed_distance = obj.sampleParticleNN(
        collapsed_particles,
        query_points
    )

    difference = np.abs(
        original_distance - collapsed_distance
    )

    relative_change = (
        np.linalg.norm(collapsed_distance - original_distance)
        / (np.linalg.norm(original_distance) + 1e-12)
    )

    print("particle count:", len(particles.positions))
    print("maximum output change:", difference.max())
    print("relative field change:", relative_change)

    color_limit = max(
        np.abs(original_distance).max(),
        np.abs(collapsed_distance).max()
    )

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    axes[0].imshow(
        original_distance,
        extent=(-extent, extent, -extent, extent),
        origin="lower",
        cmap="RdBu_r",
        vmin=-color_limit,
        vmax=color_limit
    )
    axes[0].scatter(
        obj.transform.to_world(particles.positions)[:, 0],
        obj.transform.to_world(particles.positions)[:, 1],
        color="black"
    )
    axes[0].set_title("Original particles")

    axes[1].imshow(
        collapsed_distance,
        extent=(-extent, extent, -extent, extent),
        origin="lower",
        cmap="RdBu_r",
        vmin=-color_limit,
        vmax=color_limit
    )
    axes[1].scatter(
        obj.transform.to_world(collapsed_particles.positions)[:, 0],
        obj.transform.to_world(collapsed_particles.positions)[:, 1],
        color="black"
    )
    axes[1].set_title("All particles collapsed")

    axes[2].imshow(
        difference,
        extent=(-extent, extent, -extent, extent),
        origin="lower",
        cmap="inferno"
    )
    axes[2].set_title("Absolute output difference")

    for ax in axes:
        ax.set_aspect("equal")

    plt.show()


test_particle_dependence(
    box,
    box_particles
)


In [ ]:
lattice_resolution = 31

blob_particles = particle_system.resample(blob, resolution=lattice_resolution)

print("particle count:", len(blob_particles.positions))

plot_metaballs(
    blob_particles,
    bounds=(-2.0, 2.0, -2.0, 2.0),
    resolution=500,
    sigma=0.12,
    threshold=0.0,
    transform=blob.transform
)

plot_particle_model(
    blob,
    blob_particles,
    extent=2.0,
    resolution=500
)


In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

def animate_particle_model(
    obj,
    particle_lattice_resolution=31,
    lattice_extent=1.5,
    field_extent=2.0,
    field_plot_resolution=160,
    frame_count=120,
    interval=50,
    dt=0.05,
    velocity_retention=0.92,
    brownian_kick=0.02,
    seed=0
):
    particles = create_particle_buffer(
        obj,
        extent=lattice_extent,
        lattice_resolution=particle_lattice_resolution
    )

    if len(particles.positions) == 0:
        raise RuntimeError("Particle buffer is empty")

    initial_center = particles.positions.mean(
        axis=0,
        keepdims=True
    ).copy()

    axis = np.linspace(
        -field_extent,
        field_extent,
        field_plot_resolution
    )

    x, y = np.meshgrid(axis, axis)

    query_points = np.stack(
        (x, y),
        axis=-1
    )

    initial_field = obj.sampleParticleNN(
        particles,
        query_points
    )

    fig, ax = plt.subplots(figsize=(7, 7))

    field_image = ax.imshow(
        initial_field,
        extent=(
            -field_extent,
            field_extent,
            -field_extent,
            field_extent
        ),
        origin="lower",
        cmap="RdBu_r"
    )

    particle_plot = ax.scatter(
        obj.transform.to_world(particles.positions)[:, 0],
        obj.transform.to_world(particles.positions)[:, 1],
        s=6,
        color="black"
    )

    ax.set_xlim(-field_extent, field_extent)
    ax.set_ylim(-field_extent, field_extent)
    ax.set_aspect("equal")
    ax.set_title("Particle NN field — frame 0")

    rng = np.random.default_rng(seed)

    def initialize():
        field_image.set_data(initial_field)

        field_image.set_clim(
            initial_field.min(),
            initial_field.max()
        )

        particle_plot.set_offsets(
            obj.transform.to_world(particles.positions)
        )

        ax.set_title("Particle NN field — frame 0")

        return field_image, particle_plot

    def update(frame):
        if frame > 0:
            random_kicks = rng.normal(
                size=particles.velocities.shape
            ).astype(np.float32)

            particles.velocities *= velocity_retention
            particles.velocities += (
                brownian_kick * random_kicks
            )

            particles.velocities -= (
                particles.velocities.mean(
                    axis=0,
                    keepdims=True
                )
            )

            particles.step(dt)

            current_center = particles.positions.mean(
                axis=0,
                keepdims=True
            )

            particles.positions -= (
                current_center - initial_center
            )

        predicted_field = obj.sampleParticleNN(
            particles,
            query_points
        )

        field_image.set_data(predicted_field)

        field_image.set_clim(
            predicted_field.min(),
            predicted_field.max()
        )

        particle_plot.set_offsets(
            obj.transform.to_world(particles.positions)
        )

        ax.set_title(
            f"Particle NN field — frame {frame}"
        )

        return field_image, particle_plot

    animation = FuncAnimation(
        fig,
        update,
        init_func=initialize,
        frames=frame_count,
        interval=interval,
        blit=False,
        repeat=True
    )

    animation_html = animation.to_jshtml()

    plt.close(fig)
    display(HTML(animation_html))

    return particles, animation


circle.particle_model = shared_particle_model

animation_particles, particle_animation = animate_particle_model(
    circle,
    particle_lattice_resolution=31,
    field_plot_resolution=160,
    frame_count=120,
    brownian_kick=0.02
)


In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display
from scipy.optimize import linear_sum_assignment

def animate_all_primitives(
    obj,
    particle_lattice_resolution=31,
    lattice_extent=1.5,
    field_extent=2.0,
    field_plot_resolution=128,
    transition_frames=60,
    hold_frames=25,
    scatter_frames=180,
    interval=40,
    seed=0
):
    center = np.zeros(2, dtype=np.float32)

    axis = np.linspace(
        -lattice_extent,
        lattice_extent,
        particle_lattice_resolution
    )

    x, y = np.meshgrid(axis, axis)

    lattice_points = np.stack(
        (x.ravel(), y.ravel()),
        axis=-1
    ).astype(np.float32)

    primitive_names = list(SDFS.keys())
    primitive_point_sets = []

    for name in primitive_names:
        sdf = SDFS[name]
        distances = sdf(lattice_points)

        local_inside_points = lattice_points[
            distances < 0.0
        ]

        if len(local_inside_points) == 0:
            raise RuntimeError(
                f"{name} produced no particles"
            )

        primitive_point_sets.append(
            local_inside_points + center
        )

    particle_count = min(
        len(points)
        for points in primitive_point_sets
    )

    def select_lattice_points(points):
        local_points = points - center

        order = np.lexsort(
            (
                local_points[:, 0],
                local_points[:, 1]
            )
        )

        sorted_points = points[order]

        indices = np.linspace(
            0,
            len(sorted_points) - 1,
            particle_count
        ).round().astype(int)

        return sorted_points[indices].copy()

    selected_targets = [
        select_lattice_points(points)
        for points in primitive_point_sets
    ]

    ordered_targets = [
        selected_targets[0]
    ]

    for target in selected_targets[1:]:
        previous = ordered_targets[-1]

        difference = (
            previous[:, None, :]
            - target[None, :, :]
        )

        assignment_cost = np.sum(
            difference * difference,
            axis=-1
        )

        previous_indices, target_indices = (
            linear_sum_assignment(assignment_cost)
        )

        ordered_target = np.empty_like(target)

        ordered_target[previous_indices] = (
            target[target_indices]
        )

        ordered_targets.append(ordered_target)

    frame_positions = []
    frame_titles = []

    for _ in range(hold_frames):
        frame_positions.append(
            ordered_targets[0].copy()
        )
        frame_titles.append(
            primitive_names[0]
        )

    for primitive_index in range(
        1,
        len(ordered_targets)
    ):
        start = ordered_targets[
            primitive_index - 1
        ]

        end = ordered_targets[
            primitive_index
        ]

        start_name = primitive_names[
            primitive_index - 1
        ]

        end_name = primitive_names[
            primitive_index
        ]

        for transition_frame in range(
            transition_frames
        ):
            t = (
                transition_frame + 1
            ) / transition_frames

            interpolation = (
                0.5
                - 0.5 * np.cos(np.pi * t)
            )

            positions = (
                (1.0 - interpolation) * start
                + interpolation * end
            )

            frame_positions.append(
                positions.astype(np.float32)
            )

            frame_titles.append(
                f"{start_name} → {end_name}"
            )

        for _ in range(hold_frames):
            frame_positions.append(
                end.copy()
            )
            frame_titles.append(
                end_name
            )

    rng = np.random.default_rng(seed)

    scatter_positions = ordered_targets[-1].copy()

    scatter_velocities = np.zeros_like(
        scatter_positions
    )

    for scatter_frame in range(
        scatter_frames
    ):
        relative_positions = (
            scatter_positions - center
        )

        lengths = np.linalg.norm(
            relative_positions,
            axis=1,
            keepdims=True
        )

        outward_directions = (
            relative_positions
            / np.maximum(lengths, 1e-6)
        )

        random_kicks = rng.normal(
            size=scatter_positions.shape
        ).astype(np.float32)

        scatter_velocities *= 0.96

        scatter_velocities += (
            0.00015 * outward_directions
        )

        scatter_velocities += (
            0.0008 * random_kicks
        )

        scatter_velocities -= (
            scatter_velocities.mean(
                axis=0,
                keepdims=True
            )
        )

        scatter_positions += scatter_velocities

        frame_positions.append(
            scatter_positions.copy()
        )

        frame_titles.append(
            f"Scatter {scatter_frame + 1}"
        )

    particles = ParticleBuffer()
    particles.push(frame_positions[0])

    query_axis = np.linspace(
        -field_extent,
        field_extent,
        field_plot_resolution
    )

    query_x, query_y = np.meshgrid(
        query_axis,
        query_axis
    )

    query_points = np.stack(
        (query_x, query_y),
        axis=-1
    )

    initial_field = obj.sampleParticleNN(
        particles,
        query_points
    )

    fig, ax = plt.subplots(figsize=(7, 7))

    field_image = ax.imshow(
        initial_field,
        extent=(
            -field_extent,
            field_extent,
            -field_extent,
            field_extent
        ),
        origin="lower",
        cmap="RdBu_r"
    )

    particle_plot = ax.scatter(
        obj.transform.to_world(particles.positions)[:, 0],
        obj.transform.to_world(particles.positions)[:, 1],
        s=5,
        color="black"
    )

    ax.set_xlim(-field_extent, field_extent)
    ax.set_ylim(-field_extent, field_extent)
    ax.set_aspect("equal")
    ax.set_title(frame_titles[0])

    def initialize():
        particles.positions[:] = frame_positions[0]

        field_image.set_data(initial_field)

        field_image.set_clim(
            initial_field.min(),
            initial_field.max()
        )

        particle_plot.set_offsets(
            obj.transform.to_world(particles.positions)
        )

        ax.set_title(frame_titles[0])

        return field_image, particle_plot

    def update(frame):
        particles.positions[:] = (
            frame_positions[frame]
        )

        predicted_field = obj.sampleParticleNN(
            particles,
            query_points
        )

        field_image.set_data(
            predicted_field
        )

        field_image.set_clim(
            predicted_field.min(),
            predicted_field.max()
        )

        particle_plot.set_offsets(
            obj.transform.to_world(particles.positions)
        )

        ax.set_title(
            frame_titles[frame]
        )

        return field_image, particle_plot

    animation = FuncAnimation(
        fig,
        update,
        init_func=initialize,
        frames=len(frame_positions),
        interval=interval,
        blit=False,
        repeat=True
    )

    animation_html = animation.to_html5_video()

    plt.close(fig)
    display(HTML(animation_html))

    duration = (
        len(frame_positions)
        * interval
        / 1000.0
    )

    print("Particles:", particle_count)
    print("Frames:", len(frame_positions))
    print("Duration:", duration, "seconds")

    return particles, animation


circle.particle_model = shared_particle_model

animation_particles, particle_animation = animate_all_primitives(
    circle
)
